# 02 - Dataset Creation

This notebook demonstrates the core Stage 2 data shape locally, then shows the live command that builds a full dataset from an Elasticsearch collection.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts').exists():
    for parent in Path.cwd().parents:
        if (parent / 'scripts').exists() and (parent / 'pyproject.toml').exists():
            REPO_ROOT = parent
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT = REPO_ROOT / 'tutorial' / '_outputs' / 'dataset-smoke'
OUT.mkdir(parents=True, exist_ok=True)

RUN_LIVE_STAGE2 = False
print(OUT)


Stage 0 groups HTML chunks by URL and keeps binary/document chunks per chunk. This local example uses the real grouping code with synthetic chunks.


In [ ]:
from scripts.pipeline.stage0_corpus_prep import build_passages

chunks = [
    {
        '_id': 'html-1',
        'url': 'https://docs.example.com/platform/widget/latest/install.html',
        'chunk_index': 0,
        'text': 'Runtime install requires a supported Kubernetes cluster, a container runtime, and access to the vendor registry. Configure the namespace before deploying the chart. The chart creates the service account, deployment, and service objects used by the runtime.',
        'vector': [0.1, 0.2],
        'doc_type': 'text',
        'domain_area': 'platform operations',
        'domain_slice': 'runtime deployment',
    },
    {
        '_id': 'html-2',
        'url': 'https://docs.example.com/platform/widget/latest/install.html',
        'chunk_index': 1,
        'text': 'After installation, verify readiness by checking the health endpoint and confirming that model profiles are listed. Failed pulls usually indicate registry credentials or node selector mismatches.',
        'vector': [0.1, 0.2],
        'doc_type': 'text',
        'domain_area': 'platform operations',
        'domain_slice': 'runtime deployment',
    },
    {
        '_id': 'pdf-1',
        'url': 'https://assets.example.com/platform/widget/deployment-guide.pdf',
        'chunk_index': 3,
        'text': 'For air-gapped deployments, mirror the model artifacts and container images into an internal registry. Update image pull secrets and configure the runtime to read adapters from a mounted filesystem path.',
        'vector': [0.4, 0.5],
        'doc_type': 'pdf',
        'domain_area': 'platform operations',
        'domain_slice': 'runtime deployment',
    },
]

passages = build_passages(chunks, min_passage_tokens=10)
for passage in passages:
    print(passage.model_dump_json(indent=2))


The full pipeline produces `KVPRow` records, then the curator deduplicates, filters, and writes Customizer-format `training.jsonl` and `validation.jsonl`.


In [ ]:
from scripts.pipeline.models import KVPRow
from scripts.pipeline.stage3_curator import run_stage3

rows = [
    KVPRow(
        passage_id='p1', source_url='https://docs.example.com/platform/widget/latest/install.html',
        domain_slice='runtime deployment', stage='1a', premise_index=0,
        question='What infrastructure must be available before installing the runtime on Kubernetes?',
        answer='A supported Kubernetes cluster, a container runtime, and access to the vendor registry must be available before installing the runtime. The namespace should also be configured before deploying the chart.',
        context=passages[0].text,
    ),
    KVPRow(
        passage_id='p1', source_url='https://docs.example.com/platform/widget/latest/install.html',
        domain_slice='runtime deployment', stage='1a', premise_index=1,
        question='Which Kubernetes objects does the the runtime chart create during installation?',
        answer='The the runtime chart creates the service account, deployment, and service objects used by the runtime. These objects are created after the namespace is configured and the chart is deployed.',
        context=passages[0].text,
    ),
    KVPRow(
        passage_id='p1', source_url='https://docs.example.com/platform/widget/latest/install.html',
        domain_slice='runtime deployment', stage='1b', premise_index=0,
        question='How should an operator verify that the runtime is ready after installation?',
        answer='The operator should check the health endpoint and confirm that model profiles are listed. If image pulls fail, registry credentials and node selector settings should be checked first.',
        context=passages[0].text,
        qa_type='bridging',
    ),
    KVPRow(
        passage_id='p1', source_url='https://docs.example.com/platform/widget/latest/install.html',
        domain_slice='runtime deployment', stage='1b', premise_index=1,
        question='What troubleshooting path connects failed image pulls with installation readiness?',
        answer='Failed image pulls can prevent the runtime from becoming ready, so readiness checks should be paired with registry credential and node selector validation. The health endpoint and model profile listing confirm whether the deployment recovered.',
        context=passages[0].text,
        qa_type='bridging',
    ),
    KVPRow(
        passage_id='p2', source_url='https://assets.example.com/platform/widget/deployment-guide.pdf',
        domain_slice='runtime deployment', stage='1c', premise_index=0,
        question='Summarize the required changes for an air-gapped the runtime deployment.',
        answer='For an air-gapped deployment, mirror model artifacts and container images into an internal registry. Then update image pull secrets and configure the runtime to read adapters from a mounted filesystem path.',
        context=passages[1].text,
    ),
    KVPRow(
        passage_id='p2', source_url='https://assets.example.com/platform/widget/deployment-guide.pdf',
        domain_slice='runtime deployment', stage='1c', premise_index=1,
        question='List the artifact and configuration requirements for using adapters in an air-gapped deployment.',
        answer='The deployment must mirror model artifacts and container images into an internal registry. It must also update image pull secrets and point adapter loading at a mounted filesystem path that the runtime can read.',
        context=passages[1].text,
    ),
]

system_prompt = 'You are a precise domain technical assistant. Answer based on official documentation.'
train_rows, val_rows = run_stage3(rows, OUT, system_prompt, train_ratio=0.5, min_q_tokens=4, min_a_tokens=8)
print('train rows:', len(train_rows))
print('validation rows:', len(val_rows))
print((OUT / 'training.jsonl').read_text())


For a full run, use `scripts/build_v2_dataset.py`. Start with `--dry-run`, then a smoke test with `--max-passages`, then the full collection.


In [ ]:
import subprocess

cmd = [
    sys.executable,
    'scripts/build_v2_dataset.py',
    '--collection', 'nim_curated',
    '--output', 'tutorial/_outputs/nim_curated',
    '--dry-run',
]
print(' '.join(cmd))
if RUN_LIVE_STAGE2:
    subprocess.run(cmd[:-1] + ['--max-passages', '25'], cwd=REPO_ROOT, check=True)
